In [3]:
import os
import json
import ast
import pandas as pd

In [4]:
project_dir = os.path.join(
    os.path.expanduser("~"),
    "AI_Software_Engineering_Agent"
)

extracted_dir = os.path.join(
    project_dir,
    "data",
    "extracted_code"
)

processed_dir = os.path.join(
    project_dir,
    "data",
    "processed_code"
)

os.makedirs(processed_dir, exist_ok=True)

input_path = os.path.join(
    extracted_dir,
    "code_dataset.json"
)

output_path = os.path.join(
    processed_dir,
    "code_chunks.json"
)

print("Input:", input_path)
print("Output:", output_path)

Input: C:\Users\Tushti Gupta\AI_Software_Engineering_Agent\data\extracted_code\code_dataset.json
Output: C:\Users\Tushti Gupta\AI_Software_Engineering_Agent\data\processed_code\code_chunks.json


In [5]:
with open(input_path, "r", encoding="utf-8") as f:
    dataset = json.load(f)

print("Total Python files:", len(dataset))

Total Python files: 2960


In [6]:
def create_code_chunks(record):

    code = record["code"]
    chunks = []

    try:
        tree = ast.parse(code)

    except SyntaxError as e:
        return {
            "file_id": record["id"],
            "repository": record["repository"],
            "repository_url": record["repository_url"],
            "file_name": record["file_name"],
            "file_path": record["file_path"],
            "language": record["language"],
            "parse_success": False,
            "error": str(e),
            "chunks": []
        }

    # ---------------------------------------
    # Extract imports
    # ---------------------------------------

    imports = []

    for node in ast.walk(tree):

        if isinstance(node, ast.Import):

            for alias in node.names:
                imports.append(alias.name)

        elif isinstance(node, ast.ImportFrom):

            module = node.module or ""

            for alias in node.names:
                imports.append(
                    f"{module}.{alias.name}"
                )

    # ---------------------------------------
    # Extract classes/functions
    # ---------------------------------------

    for node in tree.body:

        # ===================================
        # FUNCTION
        # ===================================

        if isinstance(
            node,
            (ast.FunctionDef, ast.AsyncFunctionDef)
        ):

            source = ast.get_source_segment(
                code,
                node
            ) or ""

            arguments = [
                arg.arg
                for arg in node.args.args
            ]

            docstring = ast.get_docstring(node) or ""

            searchable_text = f"""
Repository: {record["repository"]}
File: {record["file_name"]}
Path: {record["file_path"]}
Type: function
Function: {node.name}
Arguments: {", ".join(arguments)}
Docstring: {docstring}
Code:
{source}
""".strip()

            chunks.append({
                "chunk_type": "function",
                "name": node.name,
                "parent_class": None,
                "line_start": node.lineno,
                "line_end": getattr(
                    node,
                    "end_lineno",
                    node.lineno
                ),
                "arguments": arguments,
                "docstring": docstring,
                "code": source,
                "searchable_text": searchable_text
            })

        # ===================================
        # CLASS
        # ===================================

        elif isinstance(node, ast.ClassDef):

            class_code = ast.get_source_segment(
                code,
                node
            ) or ""

            docstring = ast.get_docstring(node) or ""

            searchable_text = f"""
Repository: {record["repository"]}
File: {record["file_name"]}
Path: {record["file_path"]}
Type: class
Class: {node.name}
Docstring: {docstring}
Code:
{class_code}
""".strip()

            chunks.append({
                "chunk_type": "class",
                "name": node.name,
                "parent_class": None,
                "line_start": node.lineno,
                "line_end": getattr(
                    node,
                    "end_lineno",
                    node.lineno
                ),
                "arguments": [],
                "docstring": docstring,
                "code": class_code,
                "searchable_text": searchable_text
            })

            # ===================================
            # METHODS
            # ===================================

            for child in node.body:

                if isinstance(
                    child,
                    (ast.FunctionDef, ast.AsyncFunctionDef)
                ):

                    method_code = ast.get_source_segment(
                        code,
                        child
                    ) or ""

                    arguments = [
                        arg.arg
                        for arg in child.args.args
                    ]

                    docstring = (
                        ast.get_docstring(child) or ""
                    )

                    searchable_text = f"""
Repository: {record["repository"]}
File: {record["file_name"]}
Path: {record["file_path"]}
Type: method
Class: {node.name}
Method: {child.name}
Arguments: {", ".join(arguments)}
Docstring: {docstring}
Code:
{method_code}
""".strip()

                    chunks.append({
                        "chunk_type": "method",
                        "name": child.name,
                        "parent_class": node.name,
                        "line_start": child.lineno,
                        "line_end": getattr(
                            child,
                            "end_lineno",
                            child.lineno
                        ),
                        "arguments": arguments,
                        "docstring": docstring,
                        "code": method_code,
                        "searchable_text": searchable_text
                    })

    return {
        "file_id": record["id"],
        "repository": record["repository"],
        "repository_url": record["repository_url"],
        "file_name": record["file_name"],
        "file_path": record["file_path"],
        "language": record["language"],
        "parse_success": True,
        "imports": imports,
        "chunks": chunks
    }

In [7]:
sample_chunks = create_code_chunks(dataset[0])

print("Repository:", sample_chunks["repository"])
print("File:", sample_chunks["file_name"])
print("Parse successful:", sample_chunks["parse_success"])
print("Number of chunks:", len(sample_chunks["chunks"]))

Repository: requests
File: setup.py
Parse successful: True
Number of chunks: 0


In [8]:
for chunk in sample_chunks["chunks"][:5]:
    
    print("\n-----------------------------")
    print("Type:", chunk["chunk_type"])
    print("Name:", chunk["name"])
    print(
        "Lines:",
        chunk["line_start"],
        "-",
        chunk["line_end"]
    )

In [10]:
if sample_chunks["chunks"]:
    
    print(
        sample_chunks["chunks"][0]["code"]
    )

In [11]:
all_chunks = []

chunk_counter = 0

for i, record in enumerate(dataset, start=1):

    result = create_code_chunks(record)

    for chunk in result["chunks"]:

        chunk_counter += 1

        chunk["chunk_id"] = (
            f"{record['repository']}_{chunk_counter:06d}"
        )

        # Add common metadata directly to chunk
        chunk["file_id"] = record["id"]
        chunk["repository"] = record["repository"]
        chunk["repository_url"] = record["repository_url"]
        chunk["file_name"] = record["file_name"]
        chunk["file_path"] = record["file_path"]
        chunk["language"] = record["language"]

        all_chunks.append(chunk)

    if i % 100 == 0:
        print(
            f"Processed {i}/{len(dataset)} files"
        )

print("Total chunks:", len(all_chunks))

Processed 100/2960 files
Processed 200/2960 files
Processed 300/2960 files
Processed 400/2960 files
Processed 500/2960 files
Processed 600/2960 files
Processed 700/2960 files
Processed 800/2960 files
Processed 900/2960 files
Processed 1000/2960 files
Processed 1100/2960 files
Processed 1200/2960 files
Processed 1300/2960 files
Processed 1400/2960 files
Processed 1500/2960 files
Processed 1600/2960 files
Processed 1700/2960 files
Processed 1800/2960 files
Processed 1900/2960 files
Processed 2000/2960 files
Processed 2100/2960 files
Processed 2200/2960 files
Processed 2300/2960 files
Processed 2400/2960 files
Processed 2500/2960 files
Processed 2600/2960 files
Processed 2700/2960 files
Processed 2800/2960 files
Processed 2900/2960 files
Total chunks: 30650


In [12]:
output_path = os.path.join(
    processed_dir,
    "code_chunks.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        all_chunks,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Saved successfully:")
print(output_path)

Saved successfully:
C:\Users\Tushti Gupta\AI_Software_Engineering_Agent\data\processed_code\code_chunks.json


In [13]:
print("Total chunks:", len(all_chunks))

print("\nFirst chunk:")
print(json.dumps(
    all_chunks[0],
    indent=4,
    ensure_ascii=False
))

Total chunks: 30650

First chunk:
{
    "chunk_type": "class",
    "name": "FlaskyStyle",
    "parent_class": null,
    "line_start": 7,
    "line_end": 86,
    "arguments": [],
    "docstring": "",
    "code": "class FlaskyStyle(Style):\n    background_color = \"#f8f8f8\"\n    default_style = \"\"\n\n    styles = {\n        # No corresponding class for the following:\n        #Text:                     \"\", # class:  ''\n        Whitespace:                \"underline #f8f8f8\",      # class: 'w'\n        Error:                     \"#a40000 border:#ef2929\", # class: 'err'\n        Other:                     \"#000000\",                # class 'x'\n\n        Comment:                   \"italic #8f5902\", # class: 'c'\n        Comment.Preproc:           \"noitalic\",       # class: 'cp'\n\n        Keyword:                   \"bold #004461\",   # class: 'k'\n        Keyword.Constant:          \"bold #004461\",   # class: 'kc'\n        Keyword.Declaration:       \"bold #004461\",   # cl

In [14]:
from collections import Counter

# ==========================================
# PHASE 2 + 3 STATISTICS
# ==========================================

total_files = len(dataset)

total_chunks = len(all_chunks)

chunk_types = Counter(
    chunk["chunk_type"]
    for chunk in all_chunks
)

repositories = Counter(
    chunk["repository"]
    for chunk in all_chunks
)

parse_failures = sum(
    1
    for record in dataset
    if not create_code_chunks(record)["parse_success"]
)

total_lines = sum(
    chunk["line_end"] - chunk["line_start"] + 1
    for chunk in all_chunks
)

avg_chunk_lines = (
    total_lines / total_chunks
    if total_chunks > 0
    else 0
)

print("=" * 60)
print("PHASE 2 + 3 STATISTICS")
print("=" * 60)

print(f"Total Python files       : {total_files}")
print(f"Total code chunks        : {total_chunks}")

print("\n--- Chunk Types ---")
print(f"Functions                : {chunk_types.get('function', 0)}")
print(f"Classes                  : {chunk_types.get('class', 0)}")
print(f"Methods                  : {chunk_types.get('method', 0)}")

print("\n--- Parsing ---")
print(f"Successfully parsed      : {total_files - parse_failures}")
print(f"Parse failures           : {parse_failures}")

print("\n--- Code Size ---")
print(f"Total chunk lines        : {total_lines}")
print(f"Average chunk size      : {avg_chunk_lines:.2f} lines")

print("\n--- Repositories ---")

for repo, count in repositories.items():
    print(f"{repo:<15}: {count}")

print("=" * 60)

PHASE 2 + 3 STATISTICS
Total Python files       : 2960
Total code chunks        : 30650

--- Chunk Types ---
Functions                : 17770
Classes                  : 3569
Methods                  : 9311

--- Parsing ---
Successfully parsed      : 2950
Parse failures           : 10

--- Code Size ---
Total chunk lines        : 590049
Average chunk size      : 19.25 lines

--- Repositories ---
requests       : 707
flask          : 919
fastapi        : 5220
pydantic       : 8156
httpx          : 1144
pytest         : 6415
black          : 2725
rich           : 1863
click          : 1353
typer          : 2148
